In [ ]:
from scipy.signal import find_peaks
from astropy.io import fits
from specutils import Spectrum1D
import matplotlib.pyplot as plt
import numpy as np
from astropy import units as u
import requests
from io import BytesIO
import os
import csv


In [ ]:
def inject_and_plot_laser(obj_id, filt, folder="/datax/scratch/emmay/galah_spectra",
                          fwhm=1.5, laser_amp_percent=20, plot=True):
    """
    Injects a Gaussian laser spike into GALAH FITS spectrum and plots it.

    Parameters:
        obj_id (str): Object ID
        filt (str): Filter (e.g. 'B', 'V', 'R', 'I')
        folder (str): Path to FITS files
        fwhm (float): FWHM of the Gaussian in pixels
        laser_amp_percent (float): Amplitude of injection as percentage of max flux
        plot (bool): Whether to show the plot

    Returns:
        wavelength (np.ndarray): Wavelength array (in Å)
        flux_injected (np.ndarray): Flux array with injected Gaussian
    """
    filename = f"{obj_id}_{filt}.fits"
    path = os.path.join(folder, filename)

    with fits.open(path) as hdul:
        header = hdul[1].header
        flux = hdul[1].data.astype(float)

    # Build wavelength axis
    crval1 = header.get('CRVAL1')
    cdelt1 = header.get('CDELT1')
    crpix1 = header.get('CRPIX1', 1)

    npix = len(flux)
    wavelength = (crval1 + (np.arange(npix) + 1 - crpix1) * cdelt1) * u.AA

    # Inject Gaussian
    sigma = fwhm / (2 * np.sqrt(2 * np.log(2)))
    center = npix // 2
    x = np.arange(npix)
    gaussian = np.exp(-0.5 * ((x - center) / sigma) ** 2)
    gaussian *= (laser_amp_percent / 100.0) * np.nanmax(flux)

    flux_injected = flux + gaussian

    # Plot
    if plot:
        threshold_mask = gaussian > 1e-4
        plt.figure(figsize=(13, 5))
        plt.plot(wavelength, flux, color='blue', linewidth=0.6, label="Original Spectrum")
        plt.plot(wavelength[threshold_mask], flux_injected[threshold_mask], color='red', linewidth=1.0, label="Injected Region")
        plt.xlabel("Wavelength (Å)")
        plt.ylabel("Flux")
        plt.title(f"Injected Spectrum — {obj_id} ({filt}) — Amplitude = {laser_amp_percent}%")
        plt.legend()
        plt.tight_layout()
        plt.show()

    return wavelength.value, flux, flux_injected, wavelength[center]


In [21]:
def get_flux_wl(obj_id, filt, folder="/datax/scratch/emmay/galah_spectra"):
    """
    Reads a GALAH FITS spectrum and returns the wavelength and flux arrays.

    Parameters:
        obj_id (str): Object ID
        filt (str): Filter (e.g. 'B', 'V', 'R', 'I')
        folder (str): Path to FITS files

    Returns:
        wavelength (np.ndarray): Wavelength array (in Å), unitless
        flux (np.ndarray): Flux array
    """
    filename = f"{obj_id}_{filt}.fits"
    path = os.path.join(folder, filename)

    with fits.open(path) as hdul:
        header = hdul[1].header
        flux = hdul[1].data.astype(float)

    # Build wavelength axis
    crval1 = header.get('CRVAL1')
    cdelt1 = header.get('CDELT1')
    crpix1 = header.get('CRPIX1', 1)

    npix = len(flux)
    wavelength = (crval1 + (np.arange(npix) + 1 - crpix1) * cdelt1) * u.AA

    return wavelength.value, flux

## Find Peak

In [ ]:
def detect_laser_peak(wavelength, flux, width_range=(0, 10), height_fraction=0.05):
    """
    Detect a narrow injected laser peak in a spectrum.

    Parameters:
    ----------
    wavelength : array-like
        Wavelength axis (same shape as flux).
    flux : array-like
        Flux values.
    width_range : tuple
        Expected width range (in pixels) for the laser peak.
    height_fraction : float
        Fraction of max flux to use as a minimum height threshold.

    Returns:
    -------
    peak_indices : array
        Indices of detected peaks.
    peak_wavelengths : array
        Wavelengths of detected peaks.
    """

    # Set dynamic threshold: 10% of max flux by default
    height_threshold = height_fraction

    # Detect peaks
    peak_indices, properties = find_peaks(
        flux,
        threshold=height_threshold,
        #width=width_range
    )

    peak_wavelengths = wavelength[peak_indices]

    # Optional: plot
    plt.figure(figsize=(12, 5))
    plt.plot(wavelength, flux, label="Spectrum")
    plt.plot(wavelength[peak_indices], flux[peak_indices], 'rx', label="Detected Peaks")
    plt.xlabel("Wavelength (Å)")
    plt.ylabel("Flux (Counts)")
    plt.title("Laser Peak Detection")
    plt.legend()
    plt.tight_layout()
    plt.show()

    return peak_indices, peak_wavelengths


In [ ]:

def measure_width_at_y(flux, wavelength, peak_idx, y_level=1):
    """Measure width of a peak at a fixed y_level using linear interpolation."""
    # Left side
    i = peak_idx
    while i > 0 and flux[i] > y_level:
        i -= 1
    if i == 0 or flux[i] > y_level:
        return None  # No valid left edge
    left = wavelength[i] + (y_level - flux[i]) / (flux[i+1] - flux[i]) * (wavelength[i+1] - wavelength[i])

    # Right side
    i = peak_idx
    while i < len(flux) - 1 and flux[i] > y_level:
        i += 1
    if i == len(flux) - 1 or flux[i] > y_level:
        return None  # No valid right edge
    right = wavelength[i-1] + (y_level - flux[i-1]) / (flux[i] - flux[i-1]) * (wavelength[i] - wavelength[i-1])

    return left, right, right - left

In [23]:
def detect_laser_peak_with_fixed_level_width(
    wavelength, 
    flux, 
    height_fraction=0.2, 
    y_level=1.0, 
    max_pixel_width=3,
    injected_wavelength=None,
    plot=True
):
    """
    Detect laser-like peaks and measure width above a fixed y-level.
    ...
    Returns
    -------
    peak_wavelengths : list of float
        Wavelengths of detected peaks.
    peak_fluxes : list of float
        Flux values at detected peaks.
    widths : list of float
        Widths measured at y_level.
    """

    flux = np.asarray(flux)
    wavelength = np.asarray(wavelength)

    height_threshold = height_fraction * max(np.abs(flux - 1))
    peak_indices, properties = find_peaks(flux, height=height_threshold*np.nanmax(flux)+1)

    if len(peak_indices) == 0:
        #print("No peaks detected.")
        if plot:
            plt.figure(figsize=(12, 4))
            plt.plot(wavelength, flux, label="Full Spectrum")
            if injected_wavelength is not None:
                plt.axvline(x=float(injected_wavelength.value), color='r', linestyle='--', linewidth=2, label="Injected Laser")
            plt.xlabel("Wavelength (Å)")
            plt.ylabel("Flux")
            plt.title("Full Spectrum (No Peaks Detected)")
            plt.legend()
            plt.tight_layout()
            plt.show()
        return [], [], []   # CHANGED: now returns 3 empty lists

    peak_wavelengths = []
    peak_fluxes = []        # NEW
    widths = []
    valid_peak_indices = []

    pixel_scale = np.median(np.diff(wavelength))

    for i, idx in enumerate(peak_indices):
        result = measure_width_at_y(flux, wavelength, idx, y_level=y_level)
        if result is not None:
            left, right, width = result
            width_in_pixels = width / pixel_scale

            if width_in_pixels > max_pixel_width:
                continue

            peak_wavelengths.append(wavelength[idx])
            peak_fluxes.append(flux[idx])      # NEW
            widths.append(width)
            valid_peak_indices.append(idx)

            if plot:
                i_min = max(0, idx - 20)
                i_max = min(len(wavelength), idx + 20)

                plt.figure(figsize=(10, 4))
                plt.plot(wavelength[i_min:i_max], flux[i_min:i_max], label="Flux")
                plt.axvline(left, color='C2', linestyle='--')
                plt.axvline(right, color='C2', linestyle='--')
                plt.hlines(y_level, left, right, color='C3', linewidth=2, label=f'Width @ y={y_level}')
                plt.plot(wavelength[idx], flux[idx], 'rx', label="Peak Center")
                plt.xlabel("Wavelength (Å)")
                plt.ylabel("Flux")
                plt.title(f"Zoomed Peak at {wavelength[idx]:.2f} Å, Width = {width:.3f} Å ({width_in_pixels:.2f} px)")
                plt.legend()
                plt.tight_layout()
                plt.show()

    if plot and valid_peak_indices:
        plt.figure(figsize=(12, 4))
        plt.plot(wavelength, flux, label="Full Spectrum")
        for idx in valid_peak_indices:
            plt.plot(wavelength[idx], flux[idx], 'rx', label="Detected Peaks")

        if injected_wavelength is not None:
            plt.axvline(x=float(injected_wavelength.value), color='black', linestyle='--', linewidth=2, label="Injected Laser")

        plt.xlabel("Wavelength (Å)")
        plt.ylabel("Flux")
        plt.title("Full Spectrum with Detected Peaks")
        plt.legend()
        plt.tight_layout()
        plt.show()

    return peak_wavelengths, peak_fluxes, widths   # CHANGED

## Find Peaks Only Loop for Multiple Objects

In [24]:
import csv
from tqdm import tqdm

folder = "/datax/scratch/emmay/galah_spectra"
fits_files = [f for f in os.listdir(folder) if f.endswith(".fits")]

results = []
errors = []  # collect errors instead of printing them inline

for fits_file in tqdm(fits_files[:4000], desc="Detecting peaks"):
    filename_no_ext = fits_file[:-5]
    obj_id = filename_no_ext[:15]
    filt = filename_no_ext[16:]

    try:
        # Step 1: Read original spectrum (no injection)
        wave, flux = get_flux_wl(obj_id, filt, folder=folder)

        # Step 2: Try to detect peaks
        peak_wavelengths, peak_fluxes, widths = detect_laser_peak_with_fixed_level_width(
            wave, flux, height_fraction=0.15, y_level=1.0, max_pixel_width=5, plot=False
        )

        # Step 3: Save results for each detected peak
        for wl, fl, w in zip(peak_wavelengths, peak_fluxes, widths):
            results.append({
                "obj_id": obj_id,
                "filt": filt,
                "peak_wavelength": wl,
                "peak_flux": fl,
                "peak_width": w
            })

    except Exception as e:
        errors.append((fits_file, str(e)))

output_path = os.path.join(folder, "1_1000_detected_peaks.csv")
with open(output_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["obj_id", "filt", "peak_wavelength", "peak_flux", "peak_width"])
    writer.writeheader()
    writer.writerows(results)

print(f"Saved {len(results)} detected peaks to {output_path}")
if errors:
    print(f"{len(errors)} files failed (see `errors` list for details)")

Detecting peaks:   0%|          | 0/4000 [00:00<?, ?it/s]

Detecting peaks: 100%|██████████| 4000/4000 [00:33<00:00, 119.13it/s]


Saved 13756 detected peaks to /datax/scratch/emmay/galah_spectra/1_1000_detected_peaks.csv
94 files failed (see `errors` list for details)


- completeness curve (intensity vs fraction recovered)
- wavelength, amplitude of injection plot --> colored injections by whether or not they were recovered
